# Yearbook Sports Pipeline — Colab (GPU) runner

Before running: **Runtime → Change runtime type → GPU** (with Colab Pro, pick the fastest available, e.g. A100/L4).

This notebook mounts your Drive (where the PDFs live), clones the repo, installs deps (with two known-needed fixes baked in — see cell 5), and runs the OCR → name-extraction pipeline over every PDF in a Drive folder, writing results back to Drive.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo

`BRANCH = 'fix_ocr'` if you've pushed that branch to GitHub (recommended — it already has the fixes). If you leave it as `'main'`, cell 5 below patches the same fix into the cloned copy automatically, so either branch works.

If the repo is private, use `https://<TOKEN>@github.com/vzhu08/yearbook-sports.git` instead.

In [ ]:
BRANCH = 'main'  # or 'fix_ocr' if you've pushed it

%cd /content
!rm -rf /content/yearbook-sports
!git clone --branch {BRANCH} https://github.com/vzhu08/yearbook-sports.git
%cd /content/yearbook-sports
REPO_DIR = '/content/yearbook-sports'

## 4. Install dependencies

Two fixes vs. a naive `pip install paddleocr`:
- Also installs `paddlex[ocr]` — plain `paddleocr` only pulls `paddlex[ocr-core]`, which is missing packages (`einops`, `ftfy`, `scikit-learn`, `tokenizers`, etc.) that `PPStructureV3` actually needs at runtime, causing a `DependencyError` on first use.
- We deliberately do NOT use `paddleocr[all]` — that extra also drags in `langchain`/`openai` for an unrelated GenAI-parsing feature this project doesn't use.

Colab currently ships CUDA 12.x. If the GPU `paddlepaddle` install fails or silently falls back to CPU, check `!nvcc --version` and pick the matching wheel from https://www.paddlepaddle.org.cn/en/install/quick

In [ ]:
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr "paddlex[ocr]" spacy rapidfuzz pymupdf
!python -m spacy download en_core_web_trf -q

## 5. Patch the grayscale-array shape bug (skip if you cloned `fix_ocr`)

`src/text_extraction.py` renders pages as 2D grayscale arrays `(H, W)` and used to hand them straight to `PPStructureV3.predict()`. The currently-installed `paddlex` expects 3D input `(H, W, C)` and crashes with `ValueError: not enough values to unpack (expected 3, got 2)`. This cell patches the file in place; it's a no-op (skips) if the fix is already present, e.g. because you cloned `fix_ocr`.

In [ ]:
path = f'{REPO_DIR}/src/text_extraction.py'
with open(path) as f:
    src = f.read()

old = '    # NOTE: pass arrays directly; PaddleOCR will handle internally\n    return ocr.predict(gray_arrays)'
new = (
    '    # PaddleX\'s current preprocessors expect (H, W, C) input; expand our\n'
    '    # single-channel grayscale arrays to 3 channels before predicting.\n'
    '    bgr_arrays = [np.repeat(g[:, :, None], 3, axis=2) for g in gray_arrays]\n'
    '    return ocr.predict(bgr_arrays)'
)

if old in src:
    src = src.replace(old, new)
    with open(path, 'w') as f:
        f.write(src)
    print('patched: grayscale -> 3-channel fix applied')
elif 'bgr_arrays' in src:
    print('already patched (cloned fix_ocr branch) — no-op')
else:
    print('WARNING: expected snippet not found — inspect src/text_extraction.py manually')

## 6. Sanity-check GPU is actually used by Paddle + spaCy

In [ ]:
import paddle
print('paddle compiled with cuda:', paddle.device.is_compiled_with_cuda())
print('paddle device:', paddle.device.get_device())

import spacy
print('spacy gpu available:', spacy.prefer_gpu())
nlp = spacy.load('en_core_web_trf')
print('pipes:', nlp.pipe_names)

## 7. Point at your PDFs and an output folder in Drive

Edit `PDF_DIR` to the Drive folder that actually holds your PDFs (e.g. `/content/drive/MyDrive/yearbooks`). `OUT_DIR` is also on Drive so results survive after the runtime disconnects.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)  # so `import src.*` resolves

PDF_DIR = Path('/content/drive/MyDrive/yearbooks')          # <-- EDIT ME
OUT_DIR = Path('/content/drive/MyDrive/yearbooks_output')   # <-- EDIT ME (created if missing)
OUT_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(PDF_DIR.glob('*.pdf'))
print(f'found {len(pdfs)} pdfs')
for p in pdfs:
    print(' -', p.name)

## 8. Run the pipeline

Calls the three step functions directly (no subprocess isolation needed here — that was only to dodge Windows DLL/thread clashes in `main.py`). Set `ENABLE_NAME_CORRECTION = True` only if you also add `data/first_names_by_decade.json` and `data/last_names_by_decade.json` to the repo (not included) — otherwise leave it `False` to skip that optional step.

In [ ]:
from src.text_extraction import extract_text
from src.name_extraction import extract_names
from src.name_correction import run_name_correction

ENABLE_NAME_CORRECTION = False

for pdf in pdfs:
    print(f'\n=== {pdf.name} ===')

    extract_text(
        pdf_path=str(pdf),
        out_dir=str(OUT_DIR),
        dpi=200,
        use_gpu=True,
    )

    extract_names(
        pdf_path=str(pdf),
        out_dir=str(OUT_DIR),
        verbose_ner=True,
    )

    run_name_correction(
        pdf_path=str(pdf),
        out_dir=str(OUT_DIR),
        year=1980,
        model_name='en_core_web_trf',
        enable_correction=ENABLE_NAME_CORRECTION,
    )

print('\nAll done.')

## 9. Spot-check an output

In [ ]:
import json

first_book = OUT_DIR / pdfs[0].stem / 'sports_sections.json'
print(json.dumps(json.load(open(first_book)), indent=2)[:2000])